# PART 3: Ranking & Filtering

The ranking functions are in `myapp/search/algorithms.py` and the metrics in `myapp/search/evaluation.py`. All queries use the conjunctive (AND) filter of Part 2; only the way the matching documents are sorted changes.

In [1]:
import json, sys

import numpy as np
import pandas as pd

sys.path.append("../..")  # repository root, to import myapp

from myapp.search.algorithms import (build_terms, conjunctive_match, create_tfidf_index, create_bm25_index, rank,
                                     score_tfidf, score_bm25, score_custom, train_word2vec, text_vector, score_word2vec)
from myapp.search.evaluation import precision_at_k, avg_precision_at_k, rr_at_k, ndcg_at_k

with open("../part_1/processed_docs.json", encoding="utf-8") as f:
    docs = json.load(f)
doc_store = {d["pid"]: d for d in docs}

# relevance labels created in Part 2 (documents that are not in the file are not relevant)
labels = pd.read_csv("../part_2/test_labels.csv")
relevant = {q: set(group[group["labels"] == 1]["pid"]) for q, group in labels.groupby("query_id")}

TEST_QUERIES = {
    3: "women cotton kurta",
    4: "men running shoes",
    5: "polo neck tshirt men",
    6: "winter hooded jacket men",
    7: "white sneakers women",
}
print(len(docs), "documents")

28080 documents


## 1. Three ways of ranking

### 1.1 Indexes

TF-IDF uses the index of Part 2. BM25 also needs the length of every document and the average length. Our score uses the BM25 score and the numerical fields of each product.

In [2]:
index, idf, doc_norms = create_tfidf_index(docs)
_, bm25_idf, doc_lengths, avg_length = create_bm25_index(docs)
mean_rating = np.mean([d["average_rating"] for d in docs if d["average_rating"] is not None])

print("Vocabulary size:", len(index))
print("Average document length:", round(avg_length, 1), "terms")
print("Mean rating (used for products without rating):", round(mean_rating, 2))
print("Products without rating:", sum(d["average_rating"] is None for d in docs))
print("Products out of stock:", sum(d["out_of_stock"] for d in docs))

Vocabulary size: 6768
Average document length: 67.1 terms
Mean rating (used for products without rating): 3.63
Products without rating: 2261
Products out of stock: 1644


### 1.2 Ranking the five queries of Part 2

For each query, the AND filter selects the documents, and each method sorts them.

In [3]:
def run_query(text):
    terms = build_terms(text)
    pids = conjunctive_match(terms, index)
    bm25 = score_bm25(terms, pids, index, bm25_idf, doc_lengths, avg_length)
    return {"TF-IDF": rank(score_tfidf(terms, pids, index, idf, doc_norms)),
            "BM25": rank(bm25),
            "Our score": rank(score_custom(terms, bm25, doc_store, mean_rating))}

results = {q: run_query(text) for q, text in TEST_QUERIES.items()}

def short_title(pid):
    title = " ".join(doc_store[pid]["title"].split())
    return title[:42] + ("..." if len(title) > 42 else "")

def compare_top(q, n=10):
    """Top n of the three methods side by side. (R) marks a relevant document"""
    table = {}
    for method, ranking in results[q].items():
        table[method] = [("(R) " if pid in relevant[q] else "") + short_title(pid) for pid, _ in ranking[:n]]
    return pd.DataFrame(table, index=range(1, n + 1))

for q, text in TEST_QUERIES.items():
    print(f"Query {q}: {text}")
    display(compare_top(q))

Query 3: women cotton kurta


,TF-IDF,BM25,Our score
1,(R) Women Solid Cotton Blend Straight Kurta (B...,(R) Women Solid Cotton Blend Straight Kurta (P...,(R) Women Kurta and Pyjama Set Cotton Blend
2,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Striped Cotton Blend Straight Kurta ...
3,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Solid Cotton Blend Straight Kurta (B...
4,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Solid Cotton Blend Straight Kurta (R...
5,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Polka Print Cotton Blend A-line Kurt...
6,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Polka Print Cotton Blend A-line Kurt...
7,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Solid Cotton Blend Straight Kurta (L...
8,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Solid Cotton Blend Straight Kurta (P...
9,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Polka Print Cotton Blend A-line Kurt...
10,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Kurta and Pyjama Set Pure Cotton,(R) Women Kurta and Pyjama Set Cotton Blend


Query 4: men running shoes


,TF-IDF,BM25,Our score
1,Running Shoes For Women (White),Cricket Shoes For Women (White),(R) Running Shoes For Men (Multicolor)
2,(R) Running Shoes For Men (Multicolor),(R) Running Shoes For Men (Multicolor),(R) Running Shoes For Men (Multicolor)
3,(R) Running Shoes For Men (Multicolor),(R) Running Shoes For Men (Multicolor),"(R) SPRINT Running Shoes For Men (Black, Grey)"
4,(R) Running Shoes For Men (Blue),Running Shoes For Women (White),"(R) SPRINT Running Shoes For Men (Blue, Grey)"
5,"(R) SPRINT Running Shoes For Men (Blue, Grey)","(R) SPRINT Running Shoes For Men (Black, Grey)",(R) Running Shoes For Men (Yellow)
6,"(R) SPRINT Running Shoes For Men (Black, Grey)","(R) SPRINT Running Shoes For Men (Blue, Grey)",(R) RS-7200 Running Shoes For Men (Navy)
7,Running Shoes For Women (Multicolor),(R) RS-7200 Running Shoes For Men (Navy),(R) Running Shoes For Men (Black)
8,"Running Shoes For Women (Blue, Orange)",(R) RS-7250 Running Shoes For Men (Black),(R) RS-7250 Running Shoes For Men (Black)
9,(R) Running Shoes For Men (Black),(R) Running Shoes For Men (Blue),"(R) RS-7050 Running Shoes For Men (Blue, Navy)"
10,Running Shoes For Women (Black),"(R) RS-7050 Running Shoes For Men (Blue, Navy)",(R) Running Shoes For Men (White)


Query 5: polo neck tshirt men


,TF-IDF,BM25,Our score
1,(R) Self Design Men Polo Neck Blue T-Shirt,(R) Color Block Men Polo Neck Blue T-Shirt,(R) Self Design Men Polo Neck Blue T-Shirt
2,(R) Self Design Men Polo Neck Dark Blue T-Shir...,(R) Solid Men Polo Neck Yellow T-Shirt,(R) Self Design Men Polo Neck Dark Blue T-Shir...
3,(R) Solid Men Polo Neck Black T-Shirt,(R) Solid Men Polo Neck Yellow T-Shirt,(R) Color Block Men Polo Neck Blue T-Shirt
4,(R) Solid Men Polo Neck Yellow T-Shirt,(R) Color Block Men Polo Neck Blue T-Shirt,(R) Solid Men Polo Neck Yellow T-Shirt
5,(R) Solid Men Polo Neck Yellow T-Shirt,(R) Self Design Men Polo Neck Blue T-Shirt,(R) Solid Men Polo Neck Yellow T-Shirt
6,(R) Color Block Men Polo Neck Green T-Shirt,(R) Self Design Men Polo Neck Dark Blue T-Shir...,(R) Color Block Men Polo Neck Blue T-Shirt
7,(R) Color Block Men Polo Neck Blue T-Shirt,(R) Color Block Men Polo Neck Green T-Shirt,(R) Color Block Men Polo Neck Green T-Shirt
8,(R) Color Block Men Polo Neck Green T-Shirt,(R) Self Design Men Polo Neck Green T-Shirt,(R) Self Design Men Polo Neck Green T-Shirt
9,(R) Color Block Men Polo Neck Blue T-Shirt,(R) Color Block Men Polo Neck Green T-Shirt,(R) Color Block Men Polo Neck Green T-Shirt
10,(R) Color Block Men Polo Neck Green T-Shirt,(R) Color Block Men Polo Neck Green T-Shirt,(R) Color Block Men Polo Neck Green T-Shirt


Query 6: winter hooded jacket men


,TF-IDF,BM25,Our score
1,Full Sleeve Printed Men Jacket,(R) Full Sleeve Striped Men Casual Jacket,(R) Full Sleeve Solid Men Casual Jacket
2,Full Sleeve Solid Men Jacket,(R) Full Sleeve Striped Men Casual Jacket,(R) Full Sleeve Solid Men Casual Jacket
3,Full Sleeve Solid Men Jacket,(R) Full Sleeve Solid Men Casual Jacket,(R) Full Sleeve Solid Men Casual Jacket
4,Full Sleeve Solid Men Jacket,(R) Full Sleeve Solid Men Casual Jacket,(R) Full Sleeve Solid Men Casual Jacket
5,Full Sleeve Solid Men Jacket,(R) Full Sleeve Solid Men Casual Jacket,(R) Full Sleeve Solid Men Casual Jacket
6,Full Sleeve Solid Men Jacket,(R) Full Sleeve Solid Men Casual Jacket,(R) Full Sleeve Solid Men Casual Jacket
7,Full Sleeve Solid Men Casual Jacket,(R) Full Sleeve Solid Men Casual Jacket,(R) Full Sleeve Solid Men Casual Jacket
8,Sleeveless Solid Men Casual Jacket,(R) Full Sleeve Solid Men Casual Jacket,Full Sleeve Printed Men Casual Jacket
9,Full Sleeve Solid Men Casual Jacket,(R) Full Sleeve Solid Men Casual Jacket,Full Sleeve Printed Men Casual Jacket
10,(R) Full Sleeve Solid Men Casual Jacket,(R) Full Sleeve Color Block Men Padded Jacket,Full Sleeve Colorblock Men Sports Jacket


Query 7: white sneakers women


,TF-IDF,BM25,Our score
1,Printed Women Round Neck White T-Shirt,(R) Sneakers For Women (White),(R) Sneakers For Women (White)
2,Women Slim Fit Solid Casual Shirt,Women Slim Fit Solid Casual Shirt,(R) Sneakers For Women (White)
3,"Solid Women Round Neck White, Blue T-Shirt",Printed Women Round Neck White T-Shirt,Solid Women Collared Neck White T-Shirt (P...
4,Printed Women Round Neck White T-Shirt,Printed Women Round Neck White T-Shirt,Printed Women Round Neck White T-Shirt
5,Solid Women Collared Neck White T-Shirt (P...,Women Peds/Footie/No-Show (Pack of 5),(R) Sneakers For Women (White)
6,"Solid Women Collared Neck White, Blue T-Sh...",Women Solid Low Cut (Pack of 4),Printed Women Round Neck White T-Shirt
7,"Solid Women Collared Neck White, Black T-S...",Women Solid Low Cut (Pack of 9),Printed Women Round Neck White T-Shirt
8,"Solid Women Collared Neck White, Black T-S...",Women Solid Low Cut (Pack of 6),Graphic Print Women V Neck White T-Shirt
9,"Solid Women Collared Neck White, Blue T-Sh...","Solid Women Round Neck White, Blue T-Shirt","Solid Women Collared Neck White, Red T-Shi..."
10,"Solid Women Collared Neck White, Red T-Shi...",Printed Women Round Neck White T-Shirt,"Solid Women Collared Neck White, Red T-Shi..."


How many documents do the methods share in their top 10?

In [4]:
def shared(a, b, n=10):
    return len({p for p, _ in a[:n]} & {p for p, _ in b[:n]})

pd.DataFrame([{"query_id": q,
               "TF-IDF & BM25": shared(r["TF-IDF"], r["BM25"]),
               "BM25 & Our score": shared(r["BM25"], r["Our score"]),
               "TF-IDF & Our score": shared(r["TF-IDF"], r["Our score"])}
              for q, r in results.items()]).set_index("query_id")

,TF-IDF & BM25,BM25 & Our score,TF-IDF & Our score
query_id,,,
3,3,1,0
4,6,7,5
5,9,10,9
6,0,6,0
7,4,2,3


In [5]:
# Why do TF-IDF and BM25 disagree so much in Query 6?
q6_pids = conjunctive_match(build_terms(TEST_QUERIES[6]), index)
groups = {"relevant": [p for p in q6_pids if p in relevant[6]],
          "not relevant": [p for p in q6_pids if p not in relevant[6]]}
for name, group in groups.items():
    print(f"{name}: {len(group)} documents | average length {np.mean([doc_lengths[p] for p in group]):.1f}"
          f" | average tf of 'hood' {np.mean([index['hood'][p] for p in group]):.2f}"
          f" | average tf of 'jacket' {np.mean([index['jacket'][p] for p in group]):.2f}")
for method in ["TF-IDF", "BM25"]:
    top = [p for p, _ in results[6][method][:10]]
    print(f"{method} top 10: average length {np.mean([doc_lengths[p] for p in top]):.1f}")

relevant: 49 documents | average length 73.6 | average tf of 'hood' 1.67 | average tf of 'jacket' 4.51
not relevant: 243 documents | average length 72.2 | average tf of 'hood' 1.13 | average tf of 'jacket' 3.14
TF-IDF top 10: average length 53.1
BM25 top 10: average length 68.7


## 2. word2vec + cosine similarity

We train a skip-gram word2vec model on the processed terms of the collection. A text is represented by the average of the vectors of its words, and the documents that pass the AND filter are sorted by the cosine between the query vector and the document vector.

In [6]:
model = train_word2vec(docs)
doc_vectors = {d["pid"]: text_vector(d["all_terms"], model) for d in docs}
print("Words with a vector:", len(model.wv), "| vector size:", model.vector_size)

# quick check of what the model learned: the most similar words
for word in ["kurta", "jacket", "shoe", "sneaker", "hood"]:
    print(word, "->", [w for w, _ in model.wv.most_similar(word, topn=6)])

Words with a vector: 4985 | vector size: 100
kurta -> ['ethnic', 'mah', 'compa', '2m', 'churidar', 'patwa']
jacket -> ['quilt', 'bomber', 'puffer', 'nehru', 'varsiti', 'sweater']
shoe -> ['sandal', 'rockfield', 'mood', 'bag', 'messeng', 'up']
sneaker -> ['complet', 'saturday', 'aviat', 'scrunch', 'attain', 'allur']
hood -> ['turtl', 'nit', 'minnion', 'cowl', 'boat', 'kangaroo']


In [7]:
for q, text in TEST_QUERIES.items():
    terms = build_terms(text)
    pids = conjunctive_match(terms, index)
    results[q]["word2vec"] = rank(score_word2vec(terms, pids, doc_vectors, model))

def top20_word2vec(q):
    return pd.DataFrame([{"rank": i + 1, "score": round(score, 3), "pid": pid, "title": short_title(pid),
                          "relevant": int(pid in relevant[q])}
                         for i, (pid, score) in enumerate(results[q]["word2vec"][:20])])

for q, text in TEST_QUERIES.items():
    print(f"Query {q}: {text}")
    display(top20_word2vec(q))

Query 3: women cotton kurta


,rank,score,pid,title,relevant
0,1,0.837,DHTFWNT67TSKCV82,SIVAA TEX Women Dhoti,0
1,2,0.836,KTAF8FKVGGDFBHAE,Women Solid Cotton Blend Pathani Kurta (Pi...,1
2,3,0.836,KTAFD3E4DTKJCRJD,Women Striped Cotton Blend Straight Kurta ...,1
3,4,0.834,KTAF8FJCZVQ7QHH9,Women Solid Cotton Blend Pathani Kurta (Bl...,1
4,5,0.832,KTAFD2U8FHZRECCF,Women Self Design Cotton Blend Straight Ku...,1
5,6,0.832,KTAFD2U8ZGHHHNYF,Women Self Design Cotton Blend Straight Ku...,1
6,7,0.832,KTAFD2UBPY9HHSY6,Women Embroidered Cotton Blend Straight Ku...,1
7,8,0.831,KTAFZ8JECS48J3AG,Women Solid Cotton Blend Straight Kurta (B...,1
8,9,0.830,KTAFD2UBHHZNSX2B,Women Embroidered Cotton Blend Straight Ku...,1
9,10,0.830,ETHFFNRG9JKZFMPN,Women Kurta and Churidar Set Cotton Blend,1


Query 4: men running shoes


,rank,score,pid,title,relevant
0,1,0.792,SHOFC585JH9RGXWG,Cricket Shoes For Women (White),0
1,2,0.731,SHOFUH9JBHVZC5TU,Sneakers For Women (Red),0
2,3,0.726,SHOFVXGR2CZRYY7W,Party Wear For Men (Red),0
3,4,0.726,SHOEP5GUQYYFKYSN,Running Shoes For Women (White),0
4,5,0.726,SHOFVXGRUGGSKCSU,Party Wear For Men (Blue),0
5,6,0.724,SHOFVXGRFFD8HMBW,Party Wear For Women (Gold),0
6,7,0.724,SHOFX2G6FHVNAHZN,Loafers For Women (Black),0
7,8,0.723,SHOF5BDSWHAMZHSU,Running Shoes For Men (White),1
8,9,0.722,SHOFVJX2TP8FRR7P,Lace Up For Men (Black),0
9,10,0.721,SHOFVGHH9YGKTX4J,Sneakers For Men (Orange),0


Query 5: polo neck tshirt men


,rank,score,pid,title,relevant
0,1,0.812,TSHFHTGUQBXMZR6W,Self Design Men Polo Neck Blue T-Shirt,1
1,2,0.811,TSHFHTGGHEHSW2VX,Color Block Men Polo Neck Blue T-Shirt,1
2,3,0.811,TSHFHTGGKMW9CUBC,Color Block Men Polo Neck Blue T-Shirt,1
3,4,0.810,TSHFHTGBHE7HGXKT,Solid Men Polo Neck Yellow T-Shirt,1
4,5,0.810,TSHFHTGBXB3ENHWH,Solid Men Polo Neck Yellow T-Shirt,1
5,6,0.810,TSHFHTGUWGNGGSRC,Self Design Men Polo Neck Dark Blue T-Shir...,1
6,7,0.809,TSHFHTGKR6WZPYK2,Color Block Men Polo Neck Green T-Shirt,1
7,8,0.809,TSHFHTGKJ4R5YZ5H,Color Block Men Polo Neck Green T-Shirt,1
8,9,0.809,TSHFHTGKXCZFQV6C,Color Block Men Polo Neck Green T-Shirt,1
9,10,0.808,TSHFHTGWHGBQPY76,Self Design Men Polo Neck Green T-Shirt,1


Query 6: winter hooded jacket men


,rank,score,pid,title,relevant
0,1,0.828,JCKFMV5XJJQMNASZ,Sleeveless Solid Men Casual Jacket,0
1,2,0.821,JCKFWYNCWHNRQ4BA,Full Sleeve Striped Men Casual Jacket,1
2,3,0.819,JCKFWYNCZZUN9QYQ,Full Sleeve Striped Men Casual Jacket,1
3,4,0.816,JCKFVVHNF422SYDA,Full Sleeve Solid Men Casual Jacket,1
4,5,0.816,JCKFVVHN9PB3UNF5,Full Sleeve Solid Men Casual Jacket,1
5,6,0.814,JCKFU9APXJ3VD5CT,Full Sleeve Solid Men Casual Jacket,1
6,7,0.813,JCKFVVHN6NSHD8HT,Full Sleeve Colorblock Men Casual Jacket,0
7,8,0.813,JCKFVVHNHGCQTGQ2,Full Sleeve Colorblock Men Casual Jacket,0
8,9,0.813,JCKFVVHNMFZBYGVK,Full Sleeve Colorblock Men Casual Jacket,0
9,10,0.813,JCKFVVHNY7TN6AZH,Full Sleeve Colorblock Men Casual Jacket,0


Query 7: white sneakers women


,rank,score,pid,title,relevant
0,1,0.728,TSHFYKZNH4Z5ETPS,"Solid Women Round Neck Dark Blue, Red T-Sh...",0
1,2,0.727,TSHFW7G3RAZ2SSX3,"Solid Women Collared Neck White, Red T-Shi...",0
2,3,0.727,TSHFW7G3U2KHARFH,"Solid Women Collared Neck White, Red T-Shi...",0
3,4,0.726,TSHFW7GFA5GFJP8Y,"Solid Women Collared Neck White, Red T-Shi...",0
4,5,0.726,TSHFW7GHFEZRWFDA,"Solid Women Collared Neck White, Red T-Shi...",0
5,6,0.726,TSHFYKZNX5ZCFKHU,"Solid Women Round Neck Blue, Red T-Shirt (...",0
6,7,0.725,TSHFW7GF2JHTWHUZ,"Solid Women Collared Neck White, Blue T-Sh...",0
7,8,0.725,TSHFW7GHBZTYWZCZ,"Solid Women Collared Neck White, Black T-S...",0
8,9,0.725,TSHFW7GYGCWUQMCC,"Solid Women Collared Neck White, Black T-S...",0
9,10,0.725,TSHFW7GFQG3M2CHW,"Solid Women Collared Neck Yellow, White T-...",0


## 3. Comparison of the four methods

We use the relevance labels of Part 2. All methods rank the same documents (the AND filter does not change), so only the order changes.

In [8]:
rows = []
for q in TEST_QUERIES:
    for method, ranking in results[q].items():
        y_true = np.array([int(pid in relevant[q]) for pid, _ in ranking])
        y_score = np.array([score for _, score in ranking])
        ideal = np.ones(len(relevant[q]))
        rows.append({"query_id": q, "method": method,
                     "P@10": precision_at_k(y_true, y_score, 10),
                     "AP@10": avg_precision_at_k(y_true, y_score, 10),
                     "RR@10": rr_at_k(y_true, y_score, 10),
                     "NDCG@10": ndcg_at_k(y_true, y_score, 10, ideal)})
metrics = pd.DataFrame(rows)

pd.options.display.float_format = "{:.3f}".format
display(metrics.pivot(index="query_id", columns="method", values="NDCG@10"))
metrics.groupby("method")[["P@10", "AP@10", "RR@10", "NDCG@10"]].mean().rename(
    columns={"AP@10": "MAP@10", "RR@10": "MRR@10"})

method,BM25,Our score,TF-IDF,word2vec
query_id,,,,
3,1.000,1.000,1.000,0.780
4,0.685,1.000,0.573,0.069
5,1.000,1.000,1.000,1.000
6,1.000,0.801,0.064,0.507
7,0.469,0.947,0.000,0.000


,P@10,MAP@10,MRR@10,NDCG@10
method,,,,
BM25,0.780,0.937,0.900,0.831
Our score,0.800,0.973,1.000,0.950
TF-IDF,0.540,0.561,0.520,0.527
word2vec,0.500,0.524,0.425,0.471


### Comments
The comparison of the methods, the justification of our score and the discussion about better representations than word2vec are in the report.